In [1]:
# Set working directory
import os
os.chdir("../../")

In [2]:
# Configure file paths

# Promoter-level chec-seq signal
chec_sumprom_glob = "sumproms/*.gz"

# NT-level chec-seq signal
HUMAN_PROM_SIG_PATTERNS = ["prom_signals/{rep_name}_signals.gz"]

# FIMO results
HUMAN_PROM_FIMO_PATH = "metadata/fimo_results/promoters/fimo.tsv"

# Nucleosome signal per base
PROM_NUC_PATH = "metadata/nucleosome_signals_per_promoter.parquet"

# Optional: Yeast data, downloadable online
cisbp_yeast_rep_path = "metadata/cisbp_yeast_rep_motifs.csv"
YEAST_PROM_FIMO_PATH = ""
YEAST_PROM_SIG_TMPL = ""

# Output directories
BINDING_OUT_DIR = "binding_score_at_motifs"
BG_OUT_DIR = "binding_score_at_motifs/background_binding_arrays"
GLOBAL_NUC_BG_OUT_DIR = "binding_score_at_motifs/background_nucleosome_arrays"

## Imports


In [3]:
import gc
import glob
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd

## Data Loading


In [4]:
# Load all human promoter summary matrices
sumprom_chec_files = glob.glob(chec_sumprom_glob)
sumprom_all = pd.concat([pd.read_parquet(x) for x in sumprom_chec_files], axis=1)

corr_cutoff = 0.895

# Keep only replicates that correlate with at least one other replicate from the same sample
def filter_reproducible(sumprom_all: pd.DataFrame, cutoff) -> pd.DataFrame:
    df = sumprom_all.copy()
    groups = pd.Series(df.columns, index=df.columns).str.rsplit("_", n=2).str[0]

    keep = []
    for _, members in groups.groupby(groups).groups.items():
        if len(members) < 2:
            continue
        corr = df[members].corr()
        np.fill_diagonal(corr.values, np.nan)
        max_corrs = corr.max(axis=1)
        reproducible = max_corrs[max_corrs >= cutoff].index.tolist()
        keep.extend(reproducible)
    return df[keep]

sumprom_filtered = filter_reproducible(sumprom_all, cutoff=corr_cutoff)
good_reps = sumprom_filtered.columns
human_samples = pd.unique(["_".join(item.split("_")[:-2]) for item in good_reps])

/tmp/ipykernel_3534758/129601638.py:25: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  human_samples = pd.unique(["_".join(item.split("_")[:-2]) for item in good_reps])


## Motif Family Definitions


In [5]:
# TF DBD family definitions
FOXK1_WT = ["FOXP3", "FOXA2", "FOXF1", "FOXL1", "FOXL2", "FOXJ2", "FOXO3", "FOXP1", "FOXP2"]
FOX_DBD_swaps = ["FOXL2_DBD_FOXO3_IDR", "FOXL2_DBD_FOXA2_IDR", "FOXL2_DBD_FOXF1_IDR", "FOXL2_DBD_FOXP3_IDR", "FOXL2_DBD_FOXP1_IDR", "FOXL2_DBD_FOXP2_IDR"]
GABPA_WT = ["ELF1", "ELF2", "ERF1", "ELK1", "ELK4", "ERG", "FLI1"]
GABPA_DBD_swaps = ["ERG_DBD_ELK4", "ERG_DBD_ELK1", "ERG_DBD_FLI1", "ERG_DBD_ERF", "ERG_DBD_ELF1", "ERG_DBD_ELF2", "ERG_SAM_DBD", "ELF1_DBD_ELK4", "ELF1_DBD_ELK1", "ELF1_DBD_FLI1", "ELF1_DBD_ERF", "ELF1_DBD_ELF2", "ELF1_DBD_ERG"]
SOX10_WT = ["SOX15", "SOX17", "SOX7", "SOX11", "SOX4", "SOX6", "SOX30", "SOX9", "SOX13", "SOX5"]
SOX_DBD_IDR = ["SOX5_IDR", "SOX6_IDR", "SOX7_IDR", "SOX9_IDR", "SOX13_IDR", "SOX17_IDR", "SOX30_IDR", "SOX5_DBD", "SOX6_DBD", "SOX7_DBD", "SOX9_DBD", "SOX13_DBD", "SOX17_DBD", "SOX30_DBD"]
HXD10_WT = ["CDX2", "HOXA11", "HOXC10", "HOXD9", "HOXA10", "HOXC9", "HOXA9", "HOXB9", "CDX4", "HOXC13"]
GATA1_WT = ["GATA3", "GATA6", "GATA4", "GATA2", "GATA5"]
BATF3_WT = ["ATF4", "FOS", "CREB5", "ATF1", "CREB1", "ATF2"]
HEY1_WT = ["MNT", "MLXIPL", "MLX", "MXD4"]
NFAC4_WT = ["NFATC4", "NFATC3"]
TF2LY_WT = ["TGIF2LX", "TGIF2LY", "TGIF1", "TGIF2"]
PO3F2_WT = ["POU2F3", "POU3F4", "POU3F1"]

dbd_fam_dict = {
    "FOXK1": FOXK1_WT,
    "FOX_DBD_swaps": FOX_DBD_swaps,
    "GABPA": GABPA_WT,
    "GABPA_DBD_swaps": GABPA_DBD_swaps,
    "SOX10": SOX10_WT,
    "SOX_DBD_IDR": SOX_DBD_IDR,
    "HXD10": HXD10_WT,
    "GATA1": GATA1_WT,
    "BATF3": BATF3_WT,
    "HEY1": HEY1_WT,
    "NFAC4": NFAC4_WT,
    "TF2LY": TF2LY_WT,
    "PO3F2": PO3F2_WT,
}

dbd_fam_representative = {
    "FOXK1": "FOXK1.H12CORE.0.PS.A",
    "FOX_DBD_swaps": "FOXK1.H12CORE.0.PS.A",
    "GABPA": "GABPA.H12CORE.0.PSM.A",
    "GABPA_DBD_swaps": "GABPA.H12CORE.0.PSM.A",
    "SOX10": "SOX10.H12CORE.1.PSM.A",
    "SOX_DBD_IDR": "SOX10.H12CORE.1.PSM.A",
    "HXD10": "HXD10.H12CORE.0.SM.B",
    "GATA1": "GATA1.H12CORE.1.PSM.A",
    "BATF3": "BATF3.H12CORE.2.SM.B",
    "HEY1": "HEY1.H12CORE.0.S.B",
    "NFAC4": "NFAC4.H12CORE.1.SM.B",
    "TF2LY": "TF2LY.H12CORE.0.SM.B",
    "PO3F2": "PO3F2.H12CORE.2.SM.B",
}

dbd_type = {
    "BATF3": "bZIP",
    "FOXK1": "Forkhead",
    "GABPA": "ETS",
    "HEY1": "bHLH",
    "HXD10": "Homeodomain",
    "NFAC4": "Rel",
    "SOX10": "Sox",
    "TF2LY": "Homeodomain TALE-Type",
    "PO3F2": "Homeodomain POU",
    "GATA1": "GATA",
}

## Motif Lookup Helpers


In [6]:
priority = {"A": 0, "B": 1, "C": 2, "D": 3}

# Make a short filesystem-safe label when needed
def _safe_name(s):
    return re.sub(r"[^A-Za-z0-9._+-]+", "_", str(s))

# Pick the best motif per prefix using the A>B>C>D priority used in the original notebook
def best_motif_by_prefix_from_fimo(fimo: pd.DataFrame):
    best = {}
    for mid in fimo["motif_id"].dropna().unique():
        pref = mid.split(".", 1)[0]
        m = re.search(r"\.([A-D])$", mid)
        rank = priority[m.group(1)] if m else 9
        if pref not in best or rank < best[pref][0]:
            best[pref] = (rank, mid)
    return {k: v for k, (r, v) in best.items()}

# Match a HOCO motif directly, or through a prefix-expanded name
def _lookup_best_hoco(best_by_prefix: dict, hoco: str):
    if hoco in best_by_prefix:
        return best_by_prefix[hoco]
    for k in best_by_prefix:
        if k.startswith(hoco + "_"):
            return best_by_prefix[k]
    return None

# Resolve the best motif ID for a sample using the HOCO lookup map
def motif_id_for_sample_hoco(sample: str, best_by_prefix: dict, genesymbol_to_hoco: dict, mismatched: dict | None = None):
    gene = sample.split("_", 1)[0]
    alt = (mismatched or {}).get(gene)
    for g in (gene, alt):
        if not g:
            continue
        mid = _lookup_best_hoco(best_by_prefix, genesymbol_to_hoco.get(g, g))
        if mid:
            return mid
    return None

genesymbol_to_hoco = {
    "ATF1": "ATF1", "ATF2": "ATF2", "ATF4": "ATF4", "CDX2": "CDX2", "CDX4": "CDX4", "CREB1": "CREB1", "CREB5": "CREB5", "ELF1": "ELF1", "ELF1_DBD_ELF2": "ELF1", "ELF1_DBD_ELK1": "ELF1", "ELF1_DBD_ELK4": "ELF1", "ELF1_DBD_ERF": "ELF1", "ELF1_DBD_ERG": "ELF1", "ELF1_DBD_FLI1": "ELF1", "ELF2": "ELF2", "ELK1": "ELK1", "ELK4": "ELK4", "ERF1": "ERF", "ERG": "ERG", "ERG_DBD_ELF1": "ERG", "ERG_DBD_ELF2": "ERG", "ERG_DBD_ELK1": "ERG", "ERG_DBD_ELK4": "ERG", "ERG_DBD_ERF": "ERG", "ERG_DBD_FLI1": "ERG", "ERG_SAM_DBD": "ERG", "FLI1": "FLI1", "FOS": "FOS",
    "FOXA2": "FOXA2", "FOXF1": "FOXF1", "FOXJ2": "FOXJ2", "FOXL1": "FOXL1", "FOXL2": "FOXL2", "FOXL2_DBD_FOXA2_IDR": "FOXL2", "FOXL2_DBD_FOXF1_IDR": "FOXL2", "FOXL2_DBD_FOXO3_IDR": "FOXL2", "FOXL2_DBD_FOXP1_IDR": "FOXL2", "FOXL2_DBD_FOXP2_IDR": "FOXL2", "FOXL2_DBD_FOXP3_IDR": "FOXL2", "FOXO3": "FOXO3", "FOXP1": "FOXP1", "FOXP2": "FOXP2", "FOXP3": "FOXP3", "GATA2": "GATA2", "GATA3": "GATA3", "GATA4": "GATA4", "GATA5": "GATA5", "GATA6": "GATA6",
    "HOXA10": "HXA10", "HOXA11": "HXA11", "HOXA9": "HXA9", "HOXB9": "HXB9", "HOXC10": "HXC10", "HOXC13": "HXC13", "HOXC9": "HXC9", "HOXD9": "HXD9", "MLX": "MLX", "MLXIPL": "MLXPL", "MNT": "MNT", "MXD4": "MAD4", "NFATC3": "NFAC3", "NFATC4": "NFAC4", "POU2F3": "PO2F3", "POU3F1": "PO3F1", "POU3F4": "PO3F4", "SOX11": "SOX11", "SOX13": "SOX13", "SOX13_DBD": "SOX13", "SOX13_IDR": "SOX13", "SOX15": "SOX15", "SOX17": "SOX17", "SOX17_DBD": "SOX17", "SOX17_IDR": "SOX17", "SOX30": "SOX30", "SOX30_DBD": "SOX30", "SOX30_IDR": "SOX30", "SOX4": "SOX4", "SOX5": "SOX5", "SOX5_DBD": "SOX5",
    "SOX5_IDR": "SOX5", "SOX6": "SOX6", "SOX6_DBD": "SOX6", "SOX6_IDR": "SOX6", "SOX7": "SOX7", "SOX7_DBD": "SOX7", "SOX7_IDR": "SOX7", "SOX9": "SOX9", "SOX9_DBD": "SOX9", "SOX9_IDR": "SOX9", "TGIF1": "TGIF1", "TGIF2": "TGIF2", "TGIF2LX": "TF2LX", "TGIF2LY": "TF2LY",
}

## Sample Table Requirements


In [7]:
config_columns = [
    "sample", "origin", "location",
    "fimo_path", "signal_path", "nuc_path",
    "motifs", "flanks", "nuc_flanks", "edge_trim",
]

# Build a table of samples to run the code
# sample_table must contain one row per scoring configuration.
# Each row must include these columns:
#   sample       -> sample name
#   origin       -> "human" or "yeast"
#   location     -> "prom"
#   fimo_path    -> FIMO table to score against
#   signal_path  -> one signal file path, or a list of replicate file paths
#   nuc_path     -> nucleosome matrix path for the same location
#   motifs       -> motif IDs to score for that row
#   flanks       -> signal flank sizes to test (I used 25 bp)
#   nuc_flanks   -> nucleosome flank sizes to test (I used 7 bp)
#   edge_trim    -> number of right-edge columns to trim before scoring (150 bp, this is a definitional artifact from the prom signal dataframe)
#
#
# Expected final shape:
#   sample_table = pd.DataFrame(rows, columns=config_columns)

## Binding Score Extraction


In [8]:
# Make a string safe to use in output filenames
def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._+-]+", "_", str(s))

# Build per-row prefix sums so window means can be computed quickly
def _prep_cumsums(df: pd.DataFrame) -> Tuple[int, np.ndarray, List[np.ndarray], List[np.ndarray]]:
    A = df.to_numpy(float)
    n_rows, n_cols = A.shape
    first_valid = np.full(n_rows, -1, int)
    S, C = [], []
    for i in range(n_rows):
        row = A[i]
        valid = ~np.isnan(row)
        if not valid.any():
            S.append(np.zeros(1)); C.append(np.zeros(1, int)); continue
        fv = int(np.argmax(valid)); first_valid[i] = fv
        filled = np.where(valid, row, 0.0)
        S.append(np.concatenate([[0.0], np.cumsum(filled)]))
        C.append(np.concatenate([[0], np.cumsum(valid.astype(int))]))
    return n_cols, first_valid, S, C

# Clip window boundaries to the valid part of each row
def _clip(a: np.ndarray, b: np.ndarray, fv: int, n_cols: int) -> Tuple[np.ndarray, np.ndarray]:
    return np.maximum(a, fv), np.minimum(b, n_cols)

# Collect background signal means for all valid windows at one flank size
def _bg_signal_means_for_flank(n_cols: int, fv_sig: np.ndarray, Ssig: List[np.ndarray], Csig: List[np.ndarray], flank: int) -> np.ndarray:
    out = []
    for i in range(len(Ssig)):
        fv = fv_sig[i]
        if fv < 0: continue
        centers = np.arange(fv, n_cols, dtype=int)
        a, b = centers - flank, centers + flank + 1
        a, b = _clip(a, b, fv, n_cols)
        cnt = (Csig[i][b] - Csig[i][a]).astype(int)
        ok = cnt > 0
        if not np.any(ok): continue
        s = (Ssig[i][b] - Ssig[i][a])[ok]
        out.append(s / cnt[ok])
    return np.concatenate(out) if out else np.empty(0, float)

# Collect background nucleosome means for all valid windows at one flank size
def _bg_nuc_means_for_flank(n_cols: int, fv_nuc: np.ndarray, Snuc: List[np.ndarray], Cnuc: List[np.ndarray], flank: int) -> np.ndarray:
    out = []
    for i in range(len(Snuc)):
        fv = fv_nuc[i]
        if fv < 0: continue
        centers = np.arange(fv, n_cols, dtype=int)
        a, b = centers - flank, centers + flank + 1
        a, b = _clip(a, b, fv, n_cols)
        cnt = (Cnuc[i][b] - Cnuc[i][a]).astype(int)
        ok = cnt > 0
        if not np.any(ok): continue
        s = (Snuc[i][b] - Snuc[i][a])[ok]
        out.append(s / cnt[ok])
    return np.concatenate(out) if out else np.empty(0, float)

# Compute aligned nucleosome and signal background windows for the same centers
def _bg_paired_means_for_flanks(
    n_cols: int,
    fv_sig: np.ndarray, Ssig: List[np.ndarray], Csig: List[np.ndarray],
    fv_nuc: np.ndarray, Snuc: List[np.ndarray], Cnuc: List[np.ndarray],
    signal_flank: int,
    nuc_flank: int,
) -> Tuple[np.ndarray, np.ndarray]:
    signal_flank = int(signal_flank)
    nuc_flank = int(nuc_flank)

    nuc_out = []
    sig_out = []

    n_rows = len(Ssig)
    for i in range(n_rows):
        fv_s = int(fv_sig[i])
        fv_n = int(fv_nuc[i])
        if fv_s < 0 and fv_n < 0:
            continue

        # Use a shared valid start so both tracks are compared at the same centers
        fv = max(fv_s, fv_n)
        if fv < 0 or fv >= n_cols:
            continue

        centers = np.arange(fv, n_cols, dtype=int)

        # Signal means at the shared centers
        a_s = centers - signal_flank
        b_s = centers + signal_flank + 1
        a_s, b_s = _clip(a_s, b_s, fv, n_cols)
        cnt_s = (Csig[i][b_s] - Csig[i][a_s]).astype(int)
        sum_s = (Ssig[i][b_s] - Ssig[i][a_s])
        sig_mean = np.full_like(sum_s, np.nan, dtype=float)
        ok_s = cnt_s > 0
        sig_mean[ok_s] = sum_s[ok_s] / cnt_s[ok_s]

        # Nucleosome means at the same centers
        a_n = centers - nuc_flank
        b_n = centers + nuc_flank + 1
        a_n, b_n = _clip(a_n, b_n, fv, n_cols)
        cnt_n = (Cnuc[i][b_n] - Cnuc[i][a_n]).astype(int)
        sum_n = (Snuc[i][b_n] - Snuc[i][a_n])
        nuc_mean = np.full_like(sum_n, np.nan, dtype=float)
        ok_n = cnt_n > 0
        nuc_mean[ok_n] = sum_n[ok_n] / cnt_n[ok_n]

        nuc_out.append(nuc_mean)
        sig_out.append(sig_mean)

    if not nuc_out:
        return np.empty(0, float), np.empty(0, float)

    return np.concatenate(nuc_out), np.concatenate(sig_out)


# Store one sample's signal and nucleosome matrices and score motif hits against them
class SampleBackground:
    def __init__(
        self,
        sample: str,
        signal_path: Union[str, List[str]],
        nuc_path: Optional[str],
        edge_trim: int,
        bg_out_dir: str,
        origin: Optional[str] = None,
        location: Optional[str] = None,
    ):
        self.sample = str(sample)
        self.origin = str(origin) if origin is not None else "NA"
        self.location = str(location) if location is not None else "NA"
        self.bg_out_dir = Path(bg_out_dir)
        self.bg_out_dir.mkdir(parents=True, exist_ok=True)

        # Load the signal matrix, or average replicate matrices if needed
        if isinstance(signal_path, (list, tuple)):
            if not signal_path:
                raise ValueError(f"No signal files provided for sample={sample}")
            sig_dfs = [pd.read_parquet(p) for p in signal_path]
            sig0 = sig_dfs[0]
            for df, p in zip(sig_dfs[1:], signal_path[1:]):
                if df.shape != sig0.shape:
                    raise ValueError(f"Signal replicate shape mismatch for sample={sample}, file={p}: {df.shape} vs {sig0.shape}")
                if not sig0.index.equals(df.index):
                    raise ValueError(f"Signal replicate index mismatch for sample={sample}, file={p}")
                if not sig0.columns.equals(df.columns):
                    raise ValueError(f"Signal replicate columns mismatch for sample={sample}, file={p}")
            sig = sum(sig_dfs) / float(len(sig_dfs))
        elif isinstance(signal_path, str):
            sig = pd.read_parquet(signal_path)
        else:
            raise TypeError(f"signal_path must be str or list[str], got {type(signal_path)} for sample={sample}")

        if edge_trim > 0:
            sig = sig.iloc[:, : max(0, sig.shape[1] - edge_trim)]
        self.sig_df = sig

        # Load the nucleosome matrix and align it to the signal matrix
        if nuc_path is not None and not (isinstance(nuc_path, float) and np.isnan(nuc_path)):
            nuc = pd.read_parquet(nuc_path)
            if edge_trim > 0:
                nuc = nuc.iloc[:, : max(0, nuc.shape[1] - edge_trim)]
            m = min(self.sig_df.shape[1], nuc.shape[1])
            self.sig_df = self.sig_df.iloc[:, :m]
            nuc = nuc.iloc[:, :m]
            common = self.sig_df.index.intersection(nuc.index)
            self.sig_df = self.sig_df.loc[common]
            nuc = nuc.loc[common]
            self.nuc_df = nuc
        else:
            self.nuc_df = None

        # Precompute fast lookup arrays for repeated windowed calculations
        self.n_cols, self.fv_sig, self.Ssig, self.Csig = _prep_cumsums(self.sig_df)
        if self.nuc_df is not None:
            _, self.fv_nuc, self.Snuc, self.Cnuc = _prep_cumsums(self.nuc_df)
        else:
            self.fv_nuc = self.Snuc = self.Cnuc = None

        self.seq_to_row: Dict[str, int] = {str(idx): i for i, idx in enumerate(self.sig_df.index)}
        self._signal_bg_cache: Dict[int, Dict[str, object]] = {}
        self._nuc_bg_sorted_cache: Dict[Tuple[int, int], Dict[str, np.ndarray]] = {}

    # Get or build the signal background for one flank size
    def get_signal_background(self, flank: int) -> Dict[str, object]:
        flank = int(flank)
        if flank in self._signal_bg_cache:
            return self._signal_bg_cache[flank]

        bg_vals = _bg_signal_means_for_flank(self.n_cols, self.fv_sig, self.Ssig, self.Csig, flank)
        if bg_vals.size == 0:
            raise ValueError(f"No background windows for sample={self.sample}, flank={flank}")

        mean = float(np.nanmean(bg_vals))
        std = float(np.nanstd(bg_vals, ddof=0))
        med = float(np.nanmedian(bg_vals))

        z = (bg_vals - mean) / std if std != 0.0 else np.full_like(bg_vals, np.nan, float)
        fract = bg_vals / med if med != 0.0 else np.full_like(bg_vals, np.nan, float)
        stacked = np.column_stack([bg_vals, z, fract]).astype(np.float32)

        # Save the raw background values together with simple normalizations
        fname = (
            f"{safe_filename(self.sample)}"
            f"__orig-{safe_filename(self.origin)}"
            f"__loc-{safe_filename(self.location)}"
            f"__signal_bg__fl{flank}.npy"
        )
        out_path = self.bg_out_dir / fname
        np.save(out_path, stacked)

        info = {"values": bg_vals, "mean": mean, "std": std, "median": med, "file": str(out_path)}
        self._signal_bg_cache[flank] = info
        return info

    # Return paired background windows sorted by nucleosome signal
    def get_sorted_nuc_background(self, signal_flank: int, nuc_flank: int) -> Tuple[np.ndarray, np.ndarray]:
        key = (int(signal_flank), int(nuc_flank))
        if key in self._nuc_bg_sorted_cache:
            d = self._nuc_bg_sorted_cache[key]
            return d["nuc_sorted"], d["sig_sorted"]

        if self.nuc_df is None:
            empty = {"nuc_sorted": np.array([], float), "sig_sorted": np.array([], float)}
            self._nuc_bg_sorted_cache[key] = empty
            return empty["nuc_sorted"], empty["sig_sorted"]

        bg_nuc, bg_sig = _bg_paired_means_for_flanks(
            self.n_cols,
            self.fv_sig, self.Ssig, self.Csig,
            self.fv_nuc, self.Snuc, self.Cnuc,
            int(signal_flank), int(nuc_flank),
        )
        if bg_nuc.size == 0:
            empty = {"nuc_sorted": np.array([], float), "sig_sorted": np.array([], float)}
            self._nuc_bg_sorted_cache[key] = empty
            return empty["nuc_sorted"], empty["sig_sorted"]

        nuc_sort_key = np.where(np.isfinite(bg_nuc), bg_nuc, np.inf)
        order = np.argsort(nuc_sort_key)
        nuc_sorted, sig_sorted = bg_nuc[order], bg_sig[order]

        self._nuc_bg_sorted_cache[key] = {"nuc_sorted": nuc_sorted, "sig_sorted": sig_sorted}
        return nuc_sorted, sig_sorted

    # Compute one window mean around a relative center using prefix sums
    def _window_mean(self, row_index: int, center_rel: int, flank: int, S: List[np.ndarray], C: List[np.ndarray], fv_arr: np.ndarray) -> float:
        fv = fv_arr[row_index]
        if fv < 0:
            return np.nan
        center = fv + int(center_rel)
        a, b = center - flank, center + flank + 1
        a, b = _clip(np.array([a]), np.array([b]), fv, self.n_cols)
        a, b = int(a[0]), int(b[0])
        if b <= a:
            return np.nan
        cnt = int(C[row_index][b] - C[row_index][a])
        if cnt == 0:
            return np.nan
        return float(S[row_index][b] - S[row_index][a]) / cnt

    # Score one motif across all of its FIMO hits
    def score_motif_hits(self, fimo_df: pd.DataFrame, motif_id: str, signal_flank: int, nuc_flanks: List[int]) -> pd.DataFrame:
        hits = fimo_df.loc[fimo_df["motif_id"] == motif_id].copy()
        if hits.empty:
            return pd.DataFrame()

        bg_info = self.get_signal_background(signal_flank)
        bg_mean, bg_std, bg_med = bg_info["mean"], bg_info["std"], bg_info["median"]
        bg_vals = bg_info["values"]
        k_near = max(1, int(0.10 * bg_vals.size))

        nuc_sorted_maps: Dict[int, Tuple[np.ndarray, np.ndarray]] = {
            int(nf): self.get_sorted_nuc_background(signal_flank, int(nf)) for nf in nuc_flanks
        }

        rows = []
        for _, r in hits.iterrows():
            seq = str(r["sequence_name"])
            row_idx = self.seq_to_row.get(seq)
            if row_idx is None:
                continue

            center_rel = (int(r["start"]) + int(r["stop"])) // 2
            wavg_sig = self._window_mean(row_idx, center_rel, signal_flank, self.Ssig, self.Csig, self.fv_sig)
            if not np.isfinite(wavg_sig):
                continue

            z = (wavg_sig - bg_mean) / bg_std if bg_std != 0.0 else np.nan
            fract = wavg_sig / bg_med if bg_med != 0.0 else np.nan

            row_dict = {
                "motif_name": r.get("motif_alt_id", r.get("motif_name", r["motif_id"])),
                "motif_id": r["motif_id"],
                "sequence_name": r["sequence_name"],
                "start": r["start"],
                "stop": r["stop"],
                "strand": r["strand"],
                "p-value": r["p-value"],
                "q-value": r["q-value"],
                "matched_sequence": r["matched_sequence"],
                "fimo_score": r["score"],
                "z_score": z,
                "fract_over_median": fract,
            }

            for nf in nuc_flanks:
                nf = int(nf)
                nuc_col = f"nuc_score_f{nf}"
                norm_med_col = f"fract_over_median_norm_f{nf}"
                norm_z_col = f"z_score_norm_f{nf}"

                if self.nuc_df is None:
                    row_dict[nuc_col] = np.nan
                    row_dict[norm_med_col] = np.nan
                    row_dict[norm_z_col] = np.nan
                    continue

                wavg_nuc = self._window_mean(row_idx, center_rel, nf, self.Snuc, self.Cnuc, self.fv_nuc)
                row_dict[nuc_col] = wavg_nuc
                if not np.isfinite(wavg_nuc):
                    row_dict[norm_med_col] = np.nan
                    row_dict[norm_z_col] = np.nan
                    continue

                nuc_sorted, sig_sorted = nuc_sorted_maps[nf]
                if nuc_sorted.size == 0:
                    row_dict[norm_med_col] = np.nan
                    row_dict[norm_z_col] = np.nan
                    continue

                k = min(k_near, nuc_sorted.size)
                idx = int(np.searchsorted(nuc_sorted, wavg_nuc))
                half = k // 2
                lo = max(0, idx - half)
                hi = min(nuc_sorted.size, lo + k)
                lo = max(0, hi - k)

                local_sig = sig_sorted[lo:hi]
                local_sig = local_sig[np.isfinite(local_sig)]
                if local_sig.size == 0:
                    row_dict[norm_med_col] = np.nan
                    row_dict[norm_z_col] = np.nan
                else:
                    local_med = float(np.median(local_sig))
                    row_dict[norm_med_col] = np.nan if local_med == 0.0 else (wavg_sig / local_med)

                    local_mean = float(np.mean(local_sig))
                    local_std = float(np.std(local_sig, ddof=0))
                    row_dict[norm_z_col] = np.nan if local_std == 0.0 else ((wavg_sig - local_mean) / local_std)

            rows.append(row_dict)

        if not rows:
            return pd.DataFrame()
        df_out = pd.DataFrame(rows)

        # Keep these columns present even if the requested flank list does not include them
        for nf in (7, 50):
            nuc_col = f"nuc_score_f{nf}"
            norm_med_col = f"fract_over_median_norm_f{nf}"
            norm_z_col = f"z_score_norm_f{nf}"
            if nuc_col not in df_out:
                df_out[nuc_col] = np.nan
            if norm_med_col not in df_out:
                df_out[norm_med_col] = np.nan
            if norm_z_col not in df_out:
                df_out[norm_z_col] = np.nan

        df_out["sample"] = self.sample
        df_out["motif"] = motif_id
        df_out["flank"] = int(signal_flank)

        col_order = [
            "motif_name", "motif_id", "sequence_name", "start", "stop", "strand",
            "p-value", "q-value", "matched_sequence", "fimo_score", "z_score",
            "fract_over_median",
            "fract_over_median_norm_f7", "fract_over_median_norm_f50",
            "z_score_norm_f7", "z_score_norm_f50",
            "nuc_score_f7", "nuc_score_f50", "sample", "motif", "flank",
        ]
        return df_out.reindex(columns=col_order + [c for c in df_out.columns if c not in col_order])


# Run the workflow one sample/location at a time to keep memory usage low
def run_binding_from_config(config_df: pd.DataFrame, binding_out_dir: str, bg_out_dir: str):
    binding_out_dir = Path(binding_out_dir)
    binding_out_dir.mkdir(parents=True, exist_ok=True)

    fimo_cache: Dict[str, pd.DataFrame] = {}

    # Track family representative motifs for filename labeling
    family_rep_set = set(dbd_fam_representative.values()) if "dbd_fam_representative" in globals() else set()

    # Process one sample/location group at a time
    for (sample, location), sub in config_df.groupby(["sample", "location"]):
        sample = str(sample)
        location = str(location)
        sub = sub.reset_index(drop=True)
        if sub.empty:
            continue

        # Read the shared settings for this sample/location block
        first = sub.iloc[0]
        origin = str(first.get("origin", "NA"))
        signal_path = first["signal_path"]
        nuc_path = first.get("nuc_path", None)
        edge_trim = int(first["edge_trim"])

        # Build one sample background object for this block
        sb = SampleBackground(
            sample=sample,
            signal_path=signal_path,
            nuc_path=nuc_path,
            edge_trim=edge_trim,
            bg_out_dir=bg_out_dir,
            origin=origin,
            location=location,
        )

        # Create the sample output folder once
        sample_out_dir = binding_out_dir / safe_filename(sample)
        sample_out_dir.mkdir(parents=True, exist_ok=True)

        # Score each requested motif/flank setup for this sample/location block
        for _, row in sub.iterrows():
            fimo_path = str(row["fimo_path"])
            motifs = row["motifs"]
            flanks = row["flanks"]
            nuc_flanks = row["nuc_flanks"]
            edge_trim_row = int(row["edge_trim"])

            # Normalize scalar settings into lists before iterating
            if isinstance(motifs, str):
                motifs = [motifs]
            if isinstance(flanks, (int, float)):
                flanks = [int(flanks)]
            if isinstance(nuc_flanks, (int, float)):
                nuc_flanks = [int(nuc_flanks)]
            if nuc_flanks is None or (isinstance(nuc_flanks, float) and np.isnan(nuc_flanks)):
                nuc_flanks = []

            motifs = [str(m) for m in motifs]
            flanks = [int(f) for f in flanks]
            nuc_flanks = [int(n) for n in nuc_flanks]

            # Load each FIMO table only once
            if fimo_path not in fimo_cache:
                sep = "\t" if fimo_path.endswith((".tsv", ".txt")) else ","
                fimo_cache[fimo_path] = pd.read_csv(fimo_path, sep=sep)
            fimo = fimo_cache[fimo_path]

            # Build a compact nucleosome-flank label for the output filename
            nuc_tag = "none" if not nuc_flanks else "-".join(str(n) for n in sorted(set(nuc_flanks)))

            for motif_id in motifs:
                # Label the motif type so the output filename stays descriptive
                if origin == "human":
                    if motif_id in family_rep_set:
                        motif_type = "family"
                    elif "yeast_cisbp" in fimo_path:
                        motif_type = "yeast_cross"
                    else:
                        motif_type = "individual"
                elif origin == "yeast":
                    if motif_id in family_rep_set:
                        motif_type = "human_family_cross"
                    else:
                        motif_type = "yeast"
                else:
                    motif_type = "unknown"

                for fl in flanks:
                    fl = int(fl)
                    df_summary = sb.score_motif_hits(fimo, motif_id, fl, nuc_flanks)
                    if df_summary.empty:
                        continue

                    # Save one summary file per sample, motif, and flank setup
                    fname = (
                        f"{safe_filename(sample)}"
                        f"__orig-{safe_filename(origin)}"
                        f"__loc-{safe_filename(location)}"
                        f"__motif-{safe_filename(motif_id)}"
                        f"__type-{motif_type}"
                        f"__fl{fl}"
                        f"__nuc-{nuc_tag}"
                        f"__trim{edge_trim_row}.csv"
                    )
                    out_path = sample_out_dir / fname
                    df_summary.to_csv(out_path, index=False)
                    print(
                        f"[sample={sample} origin={origin} loc={location}] "
                        f"motif={motif_id} ({motif_type}), flank={fl} -> {out_path}"
                    )

        # Free large arrays before moving to the next sample
        del sb
        gc.collect()


# Build and save global nucleosome background arrays from one matrix
def save_global_nuc_bg_from_path(
    nuc_matrix_path: str,
    out_dir: str,
    nuc_flanks=(7, 50),
    edge_trim: int = 150,
    dtype=np.float32,
    label: str = "",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    nuc = pd.read_parquet(nuc_matrix_path)
    if edge_trim > 0:
        nuc = nuc.iloc[:, : max(0, nuc.shape[1] - edge_trim)]
    if nuc.shape[1] == 0:
        raise ValueError("Nucleosome matrix is empty after edge trimming.")

    n_cols, fv_nuc, Snuc, Cnuc = _prep_cumsums(nuc)
    for n in nuc_flanks:
        n = int(n)
        arr = _bg_nuc_means_for_flank(n_cols, fv_nuc, Snuc, Cnuc, n)
        if arr.size == 0:
            print(f"[GLOBAL NUC BG] label={label} flank={n}: no valid windows, skipping.")
            continue
        arr = arr.astype(dtype, copy=False)
        tag = f"_{label}" if label else ""
        out_path = out_dir / f"global_nuc_bg{tag}_n{n}.npy"
        np.save(out_path, arr)
        print(f"[GLOBAL NUC BG] label={label} flank={n}: saved {arr.size} values -> {out_path}")

## Example Execution


In [9]:
# Example execution calls are left commented out on purpose.
# They can be uncommented when the final configuration table is ready.

# run_binding_from_config(sample_table, BINDING_OUT_DIR, BG_OUT_DIR)

# save_global_nuc_bg_from_path(
#     nuc_matrix_path=PROM_NUC_PATH,
#     out_dir=GLOBAL_NUC_BG_OUT_DIR,
#     nuc_flanks=(7, 50),
#     edge_trim=150,
#     label="prom",
# )